In [17]:
pip install pandas openpyxl re


Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:


# === FIND DIFFERENCES ===

In [24]:
import pandas as pd
# === CONFIGURATION ===
file1 = '/Users/jorgevilchis/Documents/Cosas Gestell/FacturasVW/Pt1/RESULTADOS/General2025ActualizadoERR.xlsx'
file2 = '/Users/jorgevilchis/Documents/Cosas Gestell/FacturasVW/Pt1/RESULTADOS/General2025Actualizado.xlsx'

In [25]:
column_to_compare = 'Leyenda'  # <-- Replace with the column you're checking

# === LOAD FILES ===
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)

# === CHECK SHAPE ===
if df1.shape != df2.shape:
    raise ValueError("Files have different shapes! Check that they align.")


In [26]:
# === FIND DIFFERENCES WHERE TEXT IS NOT CONTAINED IN EITHER DIRECTION ===
mask = ~df1[column_to_compare].astype(str).str.lower().apply(lambda x: x.strip())\
    .combine(df2[column_to_compare].astype(str).str.lower().apply(lambda x: x.strip()),
             lambda a, b: a in b or b in a)

differences = pd.DataFrame({
    'Row': df1.index[mask] + 1,
    'Value in File 1': df1[column_to_compare][mask].values,
    'Value in File 2': df2[column_to_compare][mask].values
})

# === DISPLAY RESULTS ===
if not differences.empty:
    print(f"Rows where values are not contained within each other in column '{column_to_compare}':\n")
    print(differences.to_string(index=False))
else:
    print(f"No differences found — all values are substrings of each other.")


Rows where values are not contained within each other in column 'Leyenda':

  Row                                                     Value in File 1                                                                                Value in File 2
 3044 ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA ESTOS PRODUCTOS GOZAN DE UN ORIGEN                            PREFERENCIAL DE LA UNION EUROPEA
 3045 ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA ESTOS PRODUCTOS GOZAN DE UN ORIGEN                            PREFERENCIAL DE LA UNION EUROPEA
 3046 ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA ESTOS PRODUCTOS GOZAN DE UN ORIGEN                            PREFERENCIAL DE LA UNION EUROPEA
 3047 ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA ESTOS PRODUCTOS GOZAN DE UN ORIGEN                            PREFERENCIAL DE LA UNION EUROPEA
 3048 ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA ESTOS

In [27]:
import re

# === NORMALIZE TEXT FUNCTION ===
def clean_text(s):
    return re.sub(r'\s+', ' ', str(s)).strip().lower()

# === APPLY NORMALIZATION ===
col1_clean = df1[column_to_compare].apply(clean_text)
col2_clean = df2[column_to_compare].apply(clean_text)

# === FIND NON-MATCHING ROWS (not substrings of each other) ===
mask = ~col1_clean.combine(col2_clean, lambda a, b: a in b or b in a)

# === DISPLAY DIFFERENCES ===
differences = pd.DataFrame({
    'Row': df1.index[mask] + 1,
    'factura': df1['Factura'][mask].values,
    'Value in File 1': df1[column_to_compare][mask].values,
    'Value in File 2': df2[column_to_compare][mask].values
})

print(differences['factura'].unique())
print('--'*30)

if not differences.empty:
    print(f"Rows where values are NOT substrings of each other in column '{column_to_compare}':\n")
    print(differences.to_string(index=False))
else:
    print(f"All values in column '{column_to_compare}' are substrings of each other (after whitespace normalization).")

['598104E' '598485E' '598486E' '598490E' '611563E' '611569E' '661450E'
 '663768E' '689704E' '717357E' '717361E']
------------------------------------------------------------
Rows where values are NOT substrings of each other in column 'Leyenda':

 Row factura        Value in File 1                                                      Value in File 2
3095 598104E No se encontro leyenda  ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA
3380 598485E No se encontro leyenda  ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA
3403 598486E No se encontro leyenda  ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA
3506 598490E No se encontro leyenda  ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA
4654 611563E No se encontro leyenda  ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA
4751 611569E No se encontro leyenda  ESTOS PRODUCTOS GOZAN DE UN ORIGEN PREFERENCIAL DE LA UNION EUROPEA
5331 661450E No se

In [ ]:
######
#Nuevo helper, se mando una lista de chasies y queremos ver si hay coincidencias con los archivos enviados previamente
#queremos determinar su calificacion. 

In [1]:
import pandas as pd

pathCalificar='/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Emma_George.xlsx'
patConcetrado='/home/gestell3/Desktop/TEST_TODO.xlsx'
aCalificar= pd.read_excel(pathCalificar)
concentrado= pd.read_excel(patConcetrado)

/home/gestell3/anaconda3/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)


In [14]:
concentrado.columns


Index(['Factura', 'Pais', 'Fecha', 'Auto', 'Chasis', 'J y N', 'Amount',
       'Moneda', 'Leyenda'],
      dtype='object')

In [18]:
# Keep only the first occurrence of each (Factura, Chasis) pair
df_source_indexed = concentrado.drop_duplicates(subset=['Factura', 'Chasis'])\
                               .set_index(['Factura', 'Chasis'])

# Now map
aCalificar['J o N'] = aCalificar.set_index(['FACTSEAT', 'CHASIS'])\
                                .index.map(df_source_indexed['J y N'])

# Reset index if needed
aCalificar.reset_index(inplace=True)

aCalificar.head(100)


,level_0,index,FACTSEAT,CHASIS,J o N
0,0,0,20326530,WAUAYEGY6SA068335,J
1,1,1,20326530,WAUAYAGA7SA011332,J
2,2,2,20329088,WAUBYAF3XS1038595,NaN
3,3,3,20329089,WAUAYEGY4SA076644,NaN
4,4,4,20329089,WAUAYAGA4SA011868,NaN
...,...,...,...,...,...
95,95,95,20353188,WUAB3FGY0SA907801,J
96,96,96,20353188,WUAB3FGYXSA907577,J
97,97,97,20353188,WAUGFEF56SA018705,J
98,98,98,20353189,WAUSFCF29SN040369,J


In [20]:
# Ensure columns are cleaned up (optional but highly recommended)
concentrado['Chasis'] = concentrado['Chasis'].astype(str).str.strip()
aCalificar['CHASIS'] = aCalificar['CHASIS'].astype(str).str.strip()

# Drop duplicates in concentrado to avoid multiple matches
concentrado_unique = concentrado.drop_duplicates(subset='Chasis')

# Create a mapping from Chasis to 'J y N'
chasis_to_jyn = concentrado_unique.set_index('Chasis')['J y N']

# Fill 'J o N' in aCalificar by matching 'CHASIS'
aCalificar['J o N'] = aCalificar['CHASIS'].map(chasis_to_jyn)

aCalificar.head(100)

,level_0,index,FACTSEAT,CHASIS,J o N
0,0,0,20326530,WAUAYEGY6SA068335,J
1,1,1,20326530,WAUAYAGA7SA011332,J
2,2,2,20329088,WAUBYAF3XS1038595,NaN
3,3,3,20329089,WAUAYEGY4SA076644,NaN
4,4,4,20329089,WAUAYAGA4SA011868,NaN
...,...,...,...,...,...
95,95,95,20353188,WUAB3FGY0SA907801,J
96,96,96,20353188,WUAB3FGYXSA907577,J
97,97,97,20353188,WAUGFEF56SA018705,J
98,98,98,20353189,WAUSFCF29SN040369,J


In [ ]:
prefill = pd.read_excel('/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/factComp.xlsx')
conc2024 = pd.read_excel('/home/gestell3/Desktop/ConcentradoGeneral_al2Junio.xlsx')
# Ensure columns are cleaned up (optional but highly recommended)
conc2024['Chasis'] = conc2024['Chasis'].astype(str).str.strip()
prefill['CHASIS'] = prefill['CHASIS'].astype(str).str.strip()

# Drop duplicates in concentrado to avoid multiple matches
concentrado_unique = conc2024.drop_duplicates(subset='Chasis')

# Create a mapping from Chasis to 'J y N'
chasis_to_jyn = concentrado_unique.set_index('Chasis')['J y N']

# Fill 'J o N' in aCalificar by matching 'CHASIS'
prefill['J o N'] = prefill['CHASIS'].map(chasis_to_jyn)
prefill.head

,FACTSEAT,CHASIS,J o N
0,20326530,WAUAYEGY6SA068335,J
1,20326530,WAUAYAGA7SA011332,J
2,20329088,WAUBYAF3XS1038595,NaN
3,20329089,WAUAYEGY4SA076644,NaN
4,20329089,WAUAYAGA4SA011868,NaN
...,...,...,...
95,20353188,WUAB3FGY0SA907801,J
96,20353188,WUAB3FGYXSA907577,J
97,20353188,WAUGFEF56SA018705,J
98,20353189,WAUSFCF29SN040369,J


In [27]:
prefill.count()

FACTSEAT    2321
CHASIS      2321
J o N       1884
dtype: int64

In [ ]:
# Find rows where 'J o N' is NaN or an empty string (after stripping)
mask = aCalificar['J o N'].isna() | (aCalificar['J o N'].astype(str).str.strip() == '')

# Get the corresponding FACTSEAT values
factseat_missing_jyn = aCalificar.loc[mask, 'FACTSEAT']
factseat_missing_jyn.count()

437

In [ ]:
aCalificar.to_excel('/home/gestell3/Desktop/factCompCalificado.xlsx', index=False)

2321

In [37]:
with pd.ExcelWriter('/home/gestell3/Desktop/FacturasPorCalificar.xlsx', engine='openpyxl') as writer:
    aCalificar.to_excel(writer, sheet_name='Calificados', index=False)
    factseat_missing_jyn.to_excel(writer, sheet_name='Faltantes', index=False)


In [39]:
# Find rows where 'J o N' is NaN or an empty string (after stripping)
mask = aCalificar['J o N'].isna() | (aCalificar['J o N'].astype(str).str.strip() == '')

# Get the corresponding FACTSEAT values
factseat_missing_jyn = aCalificar.loc[mask, 'FACTSEAT']
factseat_missing_jyn.shape

(437,)

In [3]:
import pandas as pd

pathCalificar='/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Emma_George.xlsx'
pathConcentrado='/home/gestell3/Desktop/TEST_TODO.xlsx'

concentrado= pd.read_excel(pathConcentrado)
aCalificar= pd.read_excel(pathCalificar)
# 1. Clean up whitespace and ensure CHASIS columns are string type
concentrado['Chasis'] = concentrado['Chasis'].astype(str).str.strip()
aCalificar['CHASIS'] = aCalificar['CHASIS'].astype(str).str.strip()

# 2. Drop duplicates to avoid ambiguous mapping
concentrado_unique = concentrado.drop_duplicates(subset='Chasis')

# 3. Create mapping from Chasis to 'J y N'
chasis_to_jyn = concentrado_unique.set_index('Chasis')['J y N']

# 4. Fill 'J o N' in aCalificar
aCalificar['J o N'] = aCalificar['CHASIS'].map(chasis_to_jyn)

# 5. Separate into two DataFrames
# Calificados = rows where 'J o N' is not null/empty
calificados = aCalificar[
    aCalificar['J o N'].notna() & (aCalificar['J o N'].astype(str).str.strip() != '')
]

# Faltantes = rows where 'J o N' is still missing
faltantes = aCalificar[
    aCalificar['J o N'].isna() | (aCalificar['J o N'].astype(str).str.strip() == '')
]

# 6. Save to Excel
output_path = '/home/gestell3/Desktop/calificaciones_2Julio.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    calificados.to_excel(writer, sheet_name='Calificados', index=False)
    faltantes.to_excel(writer, sheet_name='Faltantes', index=False)

print(f"✅ Result saved to {output_path}")


✅ Result saved to /home/gestell3/Desktop/calificaciones_2Julio.xlsx


In [4]:
#VEMOS SI REALMENTE NO TENEMOS LAS FACTURAS O SOLO NO ESTAN LOS CHASIS 
duplicated_factseats = pd.merge(
    calificados[['FACTSEAT']],
    faltantes[['FACTSEAT']],
    on='FACTSEAT',
    how='inner'
).drop_duplicates()

if not duplicated_factseats.empty:
    print("⚠️ There are FACTSEAT values present in both Calificados and Faltantes:")
    print(duplicated_factseats)
else:
    print("✅ No FACTSEAT values are duplicated between Calificados and Faltantes.")



✅ No FACTSEAT values are duplicated between Calificados and Faltantes.


In [3]:
##NEW TEST FOR BENTLEY 
invoicepath = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/18Junio/Veracruz Invoices.pdf'

import tabula
from tabula import read_pdf
import pandas as pd
import numpy as np
from archivos import archivos
import PyPDF2
import re
import itertools

import re
import pandas as pd
import PyPDF2

def PDFs_v4(file_path1):
    print('bentley')

    with open(file_path1, 'rb') as archivo_pdf:
        lector_pdf = PyPDF2.PdfFileReader(archivo_pdf, strict=False)
        num_paginas = lector_pdf.numPages
        
        list_text1 = []
        for pagina_num in range(num_paginas):
            pagina = lector_pdf.getPage(pagina_num)
            list_text1.append(pagina.extractText())
        
        # Unir todo el texto
        cadena_completa = " ".join(list_text1)

    # Dividir por cada factura usando "InvoiceNumber/Date"
    bloques = cadena_completa.split("RequirementsCurrency")
    df_final = []

    for bloque in bloques[1:]:  # omitimos el primer split si está vacío
        bloque = "RequirementsCurrency" + bloque  # para que el regex funcione igual

        # Leyenda
        if 'estos productos gozan de un origenpreferencial de la United Kingdom' in bloque:
            leyenda = 'Salvo indicacion en sentido contrario, estos productos gozan de un origen preferencial de la United Kingdom'
            J_y_N = 'J'
        else:
            leyenda = 'No cuenta con preferencias aranceralias'
            J_y_N = 'N'
        
        # Extraer datos con regex
        try:
            match_date = re.findall(r'\d{2}.\d{2}.\d{4}Chassis', bloque)[0]
            match_date2 = match_date.replace('Chassis', '').replace('20', '').replace('.', '')
            if len(match_date2) == 4:
                match_date2 = '20' + match_date2
        except:
            match_date2 = ''

        try:
            match_pais2 = re.findall(r'Manufactured in the \w{2}', bloque)[0][-2:]
        except:
            match_pais2 = ''

        try:
            match_factura2 = re.findall(r'InvoiceNumber/Date\w+', bloque)[0].replace('InvoiceNumber/Date', '')
        except:
            match_factura2 = ''

        try:
            match_moneda2 = re.findall(r'Currency \w{3}', bloque)[0].replace('Currency ', '')
        except:
            match_moneda2 = ''

        try:
            match_chasis2 = re.findall(r'VIN.\w{17}', bloque)[0].replace('VIN.', '')
        except:
            match_chasis2 = ''

        try:
            match_amount2 = re.findall(r'Final amountUSD\s+[\d.,]+', bloque)[0].replace(' ', '').replace('FinalamountUSD', '')
        except:
            match_amount2 = ''

        try:
            match_auto2 = re.findall(r'Kingdom.[_]+DescriptionPrice\s+[_]+\w{8}', bloque)[0].replace('_','').replace(' ', '').replace('Kingdom.DescriptionPrice', '')
        except:
            match_auto2 = ''

        # Crear fila de datos
        df_row = pd.DataFrame([{
            'Factura': match_factura2,
            'Pais': match_pais2,
            'Fecha': match_date2,
            'Auto': match_auto2,
            'Chasis': match_chasis2,
            'J y N': J_y_N,
            'Amount': match_amount2,
            'Moneda': match_moneda2,
            'Leyenda': leyenda
        }])

        df_final.append(df_row)

    # Concatenar todas las filas en un solo DataFrame
    df_Bentley = pd.concat(df_final, ignore_index=True)
    return df_Bentley



dfbentley = PDFs_v4(invoicepath)


bentley


In [18]:
from PDFs_v2 import PDFs_v1,PDFs_v3
from PDFs_Final_v3 import PDFs_to_excel
import PDFs_v2  
from concentrado1 import Concentrado
from concentrado2 import Concentrado2
from Estadistico import estadistico_v2
import datetime 
import tabula
import pandas as pd

### funcion PARA NUEVO FORMATO AUDI
file_path1 = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/testNuevoFormatoAudi/103418333_2_ VW de Mexiko.pdf'


print('Audi Formato nuevo')
df_PDF = tabula.read_pdf(file_path1, pages= 'all', pandas_options={'header':None})
df_PDF_concatenado = pd.concat(df_PDF, axis = 0)
print(df_PDF_concatenado)
if len(df_PDF_concatenado.columns)==3: #tiene 3 columnas
    print('tiene 3 columnas')
    if df_PDF_concatenado.iloc[0, 0] == 'Proforma Invoice No.':
            print('tiene el mismo texto!!')
            res = extract_invoice_data(df_PDF_concatenado)
            print(res)


#print(PDFs_AudiNuevo(file_path1))


Audi Formato nuevo
                            0  \
0        Proforma Invoice No.   
1                 103418333_2   
2   we shipped via seafreight   
3              Forwarder VWKL   
4          Vessel LAKE WANAKA   
5     Port of Departure EMDEN   
6             ets: 01.07.2025   
0                     Packing   
1                  1 unpacked   
2                         NaN   
3                         NaN   
4                         NaN   
5                         NaN   
6                         NaN   
7                         NaN   
8             Delivery terms:   
9              Payment terms:   
10                        NaN   
11                        NaN   

                                                    1            2  
0                                      from/Extension         Date  
1              Petra Halabrin phone: +49 841 89 46975   03.06.2025  
2                                                 NaN          NaN  
3                                           

In [9]:
df_PDF_concatenado[0][0] == 'Proforma Invoice No.' 

0     True
0    False
Name: 0, dtype: bool

In [47]:
import pandas as pd
import re

def extract_invoice_data(df):
    result = {
        'Factura': None,
        'Fecha': None,
        'Moneda': None,
        'chassis_list': [],  # list of {'chassis': ..., 'price': ..., 'model': ...}
    }

    vin_pattern = re.compile(r'\b[A-HJ-NPR-Z0-9]{17}\b')

    for i in range(len(df)):
        row = df.iloc[i]


        # === Anchor-based lookups ===
        if str(row[0]).strip() == 'Proforma Invoice No.':
            result['Factura'] = str(df.iloc[i+1, 0]).strip()

        if str(row[2]).strip() == 'Date':
            result['Fecha'] = str(df.iloc[i+1, 2]).strip()

        if 'prices in' in str(row[2]).strip().lower():
            simbolo = str(row[2]).strip().split()[-1]
            if (simbolo == '$'):
                result['Moneda'] = 'USD'
            elif (simbolo == '€'):
                result['Moneda'] = 'EUR'

        if 'Country of origin:' in str(row[1]).strip():
            pais = str(row[1]).strip().split()[-1]
            result['Pais'] = pais if pais else None





        # === Chassis block logic ===
        if str(row[1]).strip() == 'Model and Chassis No. gross-weight kgs':
            # The chassis entries begin on the next row
            j = i + 1
            while j < len(df):
                entry_row = df.iloc[j]
                cell0 = str(entry_row[1])
                vin_match = vin_pattern.search(cell0)
                if not vin_match:
                    break  # exit once we run out of VIN-like rows

                chassis = vin_match.group()
                price = str(entry_row[2]) if len(entry_row) > 2 else None

                # Search model in the next few rows
                model = None
                for k in range(j + 1, min(j + 4, len(df))):
                    next_row = str(df.iloc[k, 1])
                    if 'Model:' in next_row:
                        model = next_row.split('Model:')[-1].strip()
                        break

                result['chassis_list'].append({
                    'chassis': chassis,
                    'price': price,
                    'model': model
                })

                j += 1

                
    return result

def flatten_invoice_dict(result):
    # Start with the chassis_list
    rows = []
    for chassis_entry in result.get('chassis_list', []):
        row = {
            'Factura': result.get('Factura'),
            'Pais': result.get('Pais',),  
            'Fecha': result.get('Fecha'),
            'Auto': chassis_entry.get('model'),
            'Chasis': chassis_entry.get('chassis'),
            'J y N': '???',
            'Amount': chassis_entry.get('price'),
            'Moneda': result.get('Moneda'),

            'Leyenda': '????'
        }
        rows.append(row)

    return pd.DataFrame(rows)


def ExtraerAudiNuevoFormato(pdfPath):
    df_PDF = tabula.read_pdf(file_path1, pages= 'all', pandas_options={'header':None})
    df_PDF_concatenado = pd.concat(df_PDF, axis = 0)
    if len(df_PDF_concatenado.columns)==3: #tiene 3 columnas
        if df_PDF_concatenado.iloc[0, 0] == 'Proforma Invoice No.':
                print('Audi Nuevo Modelo de factura')
                res = extract_invoice_data(df_PDF_concatenado)
                df_export = flatten_invoice_dict(res)
                print(df_export)
                return df_export

        

resultado = ExtraerAudiNuevoFormato(file_path1)



Audi Nuevo Modelo de factura
       Factura     Pais       Fecha    Auto             Chasis J y N  \
0  103418333_2  Germany  03.06.2025  FJBAUY  WAUZZZFJ1T1000330   ???   

      Amount Moneda Leyenda  
0  25.841,63    EUR    ????  


In [ ]:
import pandas as pd
import re

def extract_invoice_dataframe(df):
    vin_pattern = re.compile(r'\b[A-HJ-NPR-Z0-9]{17}\b')
    rows = []

    factura = None
    fecha = None
    moneda = None

    for i in range(len(df)):
        row = df.iloc[i]

        # --- Factura (Proforma Invoice) ---
        if str(row[0]).strip() == 'Proforma Invoice No.':
            factura = str(df.iloc[i+1, 0]).strip()

        # --- Fecha (Date) ---
        if 'Date' in [str(cell).strip() for cell in row]:
            try:
                col_idx = row.tolist().index('Date')
                fecha = str(df.iloc[i+1, col_idx]).strip()
            except:
                pass

        # --- Moneda (Currency) ---
        if 'prices in' in str(row[2]).strip().lower():
            simbolo = str(row[2]).strip().split()[-1]
            if simbolo == '$':
                moneda = 'USD'
            elif simbolo == '€':
                moneda = 'EUR'

        # --- Chassis block logic ---
        if str(row[1]).strip() == 'Model and Chassis No. gross-weight kgs':
            j = i + 1
            while j < len(df):
                entry_row = df.iloc[j]
                cell0 = str(entry_row[1])
                vin_match = vin_pattern.search(cell0)
                if not vin_match:
                    break

                chassis = vin_match.group()
                price = str(entry_row[2]) if len(entry_row) > 2 else None

                # Look ahead for the model
                model = None
                for k in range(j + 1, min(j + 4, len(df))):
                    next_row = str(df.iloc[k, 1])
                    if 'Model:' in next_row:
                        model = next_row.split('Model:')[-1].strip()
                        break

                # Add a full row to the output list
                rows.append({
                    'Factura': factura,
                    'Fecha': fecha,
                    'Moneda': moneda,
                    'chassis': chassis,
                    'price': price,
                    'model': model
                })

                j += 1

    return pd.DataFrame(rows)

resultado = extract_invoice_dataframe()


In [1]:
1+1

2

In [11]:
import pandas as pd
pathconc2aLimpiar = '/Users/jorgevilchis/Documents/Cosas Gestell/VW_Aduanas_P1/API_Aduanas/downloads/Concentrado2.xlsx'
df_conc2 = pd.read_excel(pathconc2aLimpiar)
pathFacturas = '/Users/jorgevilchis/Documents/Cosas Gestell/VW_Aduanas_P1/API_Aduanas/downloads/facturasProcesadas.xlsx'
dfFacturas = pd.read_excel(pathFacturas)

In [7]:
df_conc2.shape,dfFacturas.shape

((49430, 19), (37250, 9))

In [14]:

blanksconc2=df_conc2[df_conc2['J y N'].isnull() | (df_conc2['J y N'] == '')]
blanksconc2.drop_duplicates(subset=['FACT'], inplace=True)

/var/folders/wr/jfs_l3jx3b18f2q2sl8wv4_r0000gn/T/ipykernel_34221/3986070935.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blanksconc2.drop_duplicates(subset=['FACT'], inplace=True)


In [18]:
ListaFaltantes = blanksconc2[['FACT','FECFACT']].copy()
ListaFaltantes = ListaFaltantes.rename(columns={'FACT': 'Factura', 'FECFACT': 'Fecha'})
ListaFaltantes.to_excel('/Users/jorgevilchis/Documents/Cosas Gestell/Resultados para Junta 18 Julio /EnviosPostJunta/FacturasFaltantes.xlsx', index=False)

In [24]:

import tabula-py
import pandas as pd
import numpy as np
from archivos import archivos
import PyPDF2
import re
import itertools



def PDFs_v1(file_path1):
    print('audio o vw') 
    df_PDF = tabula.read_pdf(file_path1, pages= 'all', pandas_options={'header':None})
    df_PDF_concatenado = pd.concat(df_PDF, axis = 0)
    print(df_PDF_concatenado)
    # Esta condicion es para ArgentinaP
    if len(df_PDF_concatenado.columns) == 3:
        list_col2 = list(df_PDF[0][2])
        factura = list_col2[0][-8:]
        date = list_col2[2].replace('FECHA: ', '').replace('20', '').replace('/', '')

        if len(date) == 4:
            date = '20' + date

        list_col0 = list(df_PDF[0][0])
        if 'Argentina' in list_col0[1]:
            pais = ['ARG']
            J_y_N = ['CUPO']
            list_leyenda_depurado_junto = 'NO EXISTEN CONVENIOS QUE PERMITAN ALTERACIONES DE PRECIOS.'
            list_moneda = ['USD']

        list_col0_concatenado = list(df_PDF_concatenado[0])
        list_col2_concatenado = list(df_PDF_concatenado[2])

        list_chasis = []
        list_auto = []
        list_amount = []
        for i in range(len(list_col0_concatenado)):
            if '•' in str(list_col0_concatenado[i]) and str(list_col2_concatenado[i]) != 'nan':
                list_auto.append(list_col0_concatenado[i][2:8])
                list_chasis.append(list_col0_concatenado[i][9:26])
                list_amount.append(list_col2_concatenado[i].split(' ')[-1])

        list_J_N = J_y_N*len(list_auto)
        list_pais = pais*len(list_auto)

    else:
        fila_1 = df_PDF_concatenado.iloc[0]
        #print('--------------------------',str(fila_1[1]))
        # Se ajusta por si la factura vive en otra columna y con otra leyenda
        if 'Invoice No.: ' in str(fila_1[1]):
            factura = fila_1[1].replace('Invoice No.: ', '')
            #print(factura)
            # Se condiciona cuando la fecha vive en otra columna
            try:
                date = fila_1[3].replace('20', '').replace('.', '')
                #print(date)
            except:
                date = fila_1[2].replace('20', '').replace('.', '').replace('Date: ', '')
        if 'Factura: ' in str(fila_1[0]):
            factura = fila_1[0].replace('Factura: ', '')
            date = fila_1[2].replace('20', '').replace('.', '')

        if len(date) == 4:
            date = '20' + date

        # A veces la primera hoja tiene 6 columnas y el resto 7 columnas, se tratara por separado la primera hoja y luego el resto
        # 1ra hoja:
        if len(df_PDF[0].columns) != len(df_PDF_concatenado.columns):
            if len(df_PDF[0].columns) == 6:
                df_PDF_concatenado2 = pd.concat(df_PDF[1:], axis = 0)

                list_col0_hoja0 = list(df_PDF[0][0])
                list_col1_hoja0 = list(df_PDF[0][1])
                list_col6_hoja0 = list(df_PDF[0][5])

                list_col0_demas_hojas = list(df_PDF_concatenado2[0])
                list_col1_demas_hojas = list(df_PDF_concatenado2[1])
                list_col6_demas_hojas = list(df_PDF_concatenado2[6])

                list_col0 = list_col0_hoja0 + list_col0_demas_hojas
                list_col1 = list_col1_hoja0 + list_col1_demas_hojas
                list_col6 = list_col6_hoja0 + list_col6_demas_hojas

                # Aseguro que sea solo numero para poder usarlo como condicion
                list_col6_depurando = []
                list_col6_depurando2 = []
                for i in range(len(list_col6)):
                    list_col6_depurando.append(str(list_col6[i]))
                    list_col6_depurando2.append(list_col6_depurando[i].replace('.', '').replace(',', ''))

                list_auto_chasis = []
                list_auto = []
                list_chasis = []
                list_amount = []
                list_J_N = []
                list_pais = []
                for i in range(len(list_col6_depurando2)):
                    if str(list_col6_depurando2[i]) != 'nan' and len(str(list_col0[i])) > 23 and list_col6_depurando2[i].isnumeric() == True and 'EXTENDED WARRANTY - REVERSE CHARGE' not in str(list_col0[i]):
                        list_auto_chasis.append(list_col0[i])
                        list_auto.append(list_col0[i][:6])
                        list_chasis.append(list_col0[i][7:24])
                        list_amount.append(list_col6[i])
                    if str(list_col0[i]) == 'J' or str(list_col0[i]) == 'N':
                        list_J_N.append(list_col0[i])
                        list_pais.append(list_col1[i][:3])
                    # Se agrega condicion cuando es vw-IND
                    if str(list_col1[i]) == 'IND':
                        list_J_N.append('CUPO')
                        list_pais.append(list_col1[i][:3])

            # A veces la primera hoja tiene 7 columnas y el resto 8 columnas, se tratara por separado la primera hoja y luego el resto
            elif len(df_PDF[0].columns) == 7:
                df_PDF_concatenado2 = pd.concat(df_PDF[1:], axis = 0)

                list_col0_hoja0 = list(df_PDF[0][0])
                list_col1_hoja0 = list(df_PDF[0][1])
                list_col6_hoja0 = list(df_PDF[0][6])

                list_col0_demas_hojas = list(df_PDF_concatenado2[0])
                list_col1_demas_hojas = list(df_PDF_concatenado2[1])
                list_col6_demas_hojas = list(df_PDF_concatenado2[7])

                list_col0 = list_col0_hoja0 + list_col0_demas_hojas
                list_col1 = list_col1_hoja0 + list_col1_demas_hojas
                list_col6 = list_col6_hoja0 + list_col6_demas_hojas

                # Aseguro que sea solo numero para poder usarlo como condicion
                list_col6_depurando = []
                list_col6_depurando2 = []
                for i in range(len(list_col6)):
                    list_col6_depurando.append(str(list_col6[i]))
                    list_col6_depurando2.append(list_col6_depurando[i].replace('.', '').replace(',', ''))

                list_auto_chasis = []
                list_auto = []
                list_chasis = []
                list_amount = []
                list_J_N = []
                list_pais = []
                for i in range(len(list_col6_depurando2)):
                    if str(list_col6_depurando2[i]) != 'nan' and len(str(list_col0[i])) > 23 and list_col6_depurando2[i].isnumeric() == True and 'EXTENDED WARRANTY - REVERSE CHARGE' not in str(list_col0[i]):
                        list_auto_chasis.append(list_col0[i])
                        list_auto.append(list_col0[i][:6])
                        list_chasis.append(list_col0[i][7:24])
                        list_amount.append(list_col6[i])
                    if str(list_col0[i]) == 'J' or str(list_col0[i]) == 'N':
                        list_J_N.append(list_col0[i])
                        list_pais.append(list_col1[i][:3])
                    # Se agrega condicion cuando es vw-IND
                    if str(list_col1[i]) == 'IND':
                        list_J_N.append('CUPO')
                        list_pais.append(list_col1[i][:3])
        # Demas hojas
        else:
            # Condicion para cuando sean 7 o 6 columnas o 8
            if len(df_PDF_concatenado.columns) == 7:
                #print('list_col0')
                list_col0 = list(df_PDF_concatenado[0])
                print(list_col0)
                #print('--------')

                list_col1 = list(df_PDF_concatenado[1])
                #print('list_col1')
                print(list_col1)
                #print('--------')
                #print('list_col6')
                list_col6 = list(df_PDF_concatenado[6])
                list_col5 = list(df_PDF_concatenado[5])

                #print('--------')
                #print(list_col6)
                #print('--------')

            elif len(df_PDF_concatenado.columns) == 6:
                print('cols num: 6')

                list_col0 = list(df_PDF_concatenado[0])
                list_col1 = list(df_PDF_concatenado[1])
                list_col6 = list(df_PDF_concatenado[5])
            elif len(df_PDF_concatenado.columns) == 8:
                print('cols num: 8')

                list_col0 = list(df_PDF_concatenado[0])
                list_col1 = list(df_PDF_concatenado[1])
                list_col6 = list(df_PDF_concatenado[7])
            print('lista_col6 len',len(list_col6))




                



            # Aseguro que sea solo numero para poder usarlo como condicion
            list_col6_depurando = []
            list_col6_depurando2 = []
            for i in range(len(list_col6)):
                list_col6_depurando.append(str(list_col6[i]))
                list_col6_depurando2.append(list_col6_depurando[i].replace('.', '').replace(',', ''))
                #print(i,'list_col6_depurando2 ',list_col6_depurando2[i])


            list_auto_chasis = []
            list_auto = []
            list_chasis = []
            list_amount = []
            list_J_N = []
            list_pais = []


            ####CONDIION PARA BELGICA 
            if 'BEL' in str(list_col1):
                print('entramos a belgica')

                list_col5_depurando = []
                list_col5_depurando2 = []
                for i in range(len(list_col5)):
                    list_col5_depurando.append(str(list_col5[i]))
                    list_col5_depurando2.append(list_col5_depurando[i].replace('.', '').replace(',', ''))
                    print(i,'list_col5_depurando2 ',list_col5_depurando2[i])

                #print('factura',factura)
                for i in range(len(list_col0)): 
                    if  str(list_col5_depurando2[i]) != 'nan' and len(str(list_col0[i])) > 23 and list_col5_depurando2[i].isnumeric() ==True:
                        print(i,' ---->  col0[i]: ',list_col0[i])
                        #print(i,' ---->  col6[i]: ',list_col6_depurando2[i])
                        #print(i,' ---->  col1[i]: ',list_col1[i])
                        print(i,' ---->  col5[i]: ',list_col5[i])




                        list_auto_chasis.append(list_col0[i])
                        list_auto.append(list_col0[i][:6])
                        list_chasis.append(list_col0[i][7:24])
                        list_amount.append(list_col5[i])
                    else: print(i,' col0[i]: ',list_col0[i])

                
                for i in range(len(list_col6)):
                    print(i,' col6[i]: ',list_col6[i])

                for i in range(len(list_col1)):
                    print(i,' col1[i]: ',list_col1[i])


            for i in range(len(list_col6_depurando2)):
                #print(i,' col0[i]: ',list_col6_depurando2[i])
                if str(list_col6_depurando2[i]) != 'nan' and len(str(list_col0[i])) > 23 and list_col6_depurando2[i].isnumeric() == True and 'EXTENDED WARRANTY - REVERSE CHARGE' not in str(list_col0[i]):
                    print(i,' ---->  col0[i]: ',list_col0[i])
                    print(i,' ---->  col6[i]: ',list_col6[i])


                    list_auto_chasis.append(list_col0[i])
                    list_auto.append(list_col0[i][:6])
                    list_chasis.append(list_col0[i][7:24])
                    list_amount.append(list_col6[i])
                if str(list_col0[i]) == 'J' or str(list_col0[i]) == 'N' and len(list_auto_chasis)>0:
                    #print(i,' ---->  [J y N} col0[i]: ',list_col0[i])

                    #print(i,' col0[i]: ',list_col0[i])

                    list_J_N.append(list_col0[i])
                    list_pais.append(list_col1[i][:3])
                # Se agrega condicion cuando es vw-IND
                if str(list_col1[i]) == 'IND':
                    list_J_N.append('CUPO')
                    list_pais.append(list_col1[i][:3])

            # Se agrega otra condicion porque en ocasiones la informacion vive en la columna 5
            if len(list_auto_chasis) != len(list_pais):
                list_col0 = list(df_PDF_concatenado[0])
                list_col1 = list(df_PDF_concatenado[1])
                list_col6 = list(df_PDF_concatenado[5])

                for i in range(len(list_col6)):
                    if str(list_col6[i]) != 'nan' and len(str(list_col0[i])) > 23 and type(list_col1[i]) == float:
                        list_auto_chasis.append(list_col0[i])
                        list_auto.append(list_col0[i][:6])
                        list_chasis.append(list_col0[i][7:24])
                        list_amount.append(list_col6[i])

        # Obteniendo moneda
        # Condicion para cuando sean 7 o 6 columnas
        if len(df_PDF[0].columns) == 7:
            list_col4_hoja1 = list(df_PDF[0][4])
            list_col5_hoja1 = list(df_PDF[0][5])
            list_col6_hoja1 = list(df_PDF[0][6])
        elif len(df_PDF[0].columns) == 6:
            list_col4_hoja1 = list(df_PDF[0][3])
            list_col5_hoja1 = list(df_PDF[0][4])
            list_col6_hoja1 = list(df_PDF[0][5])
        elif len(df_PDF_concatenado.columns) == 8:
            list_col4_hoja1 = list(df_PDF[0][5])
            list_col5_hoja1 = list(df_PDF[0][6])
            list_col6_hoja1 = list(df_PDF[0][7])
            

        # Se utilizan dos columnas donde puede vivir la leyenda -CURRENCY- o -MONEDA-
        list_moneda = []
        for i in range(len(list_col5_hoja1)):
            if str(list_col5_hoja1[i]) == 'CURRENCY:' or str(list_col4_hoja1[i]) == 'CURRENCY:' or str(list_col5_hoja1[i]) == 'MONEDA' or str(list_col4_hoja1[i]) == 'MONEDA':
                list_moneda.append(list_col6_hoja1[i])

        # Obteniendo leyenda, se utilizara como pivote la leyenda -SHIPPING MARKS- o -DECLARAMOS BAJO JURAMENTO- o -THESE ITEMS ARE CONTROLLED- y se toman las ultimas dos hojas
        df_PDF_lastpages_concatenado = pd.concat(df_PDF[-2:], axis = 0)
        list_col0_lastpage = list(df_PDF_lastpages_concatenado[0])
        list_leyenda = []
        for i in range(len(list_col0_lastpage)):
            if 'SHIPPING MARKS' in str(list_col0_lastpage[i]) or 'DECLARAMOS BAJO JURAMENTO' in str(list_col0_lastpage[i]) or 'THESE ITEMS ARE CONTROLLED' in str(list_col0_lastpage[i]):
                index = i

        # Este try es para SUDAFRICA
        try:
            list_leyenda = list_col0_lastpage[index:]
            list_leyenda_depurado = []
            for i in range(len(list_leyenda)):
                if str(list_leyenda[i]) != 'nan':
                    list_leyenda_depurado.append(list_leyenda[i])

            list_leyenda_depurado_junto = ' '.join(list_leyenda_depurado)
        except:
            list_leyenda_depurado_junto = 'No existen convenios Aranceralios'

        # SE AGREGA CONDICION SI NUESTRAS LISTAS DE -PAIS- y -J y N- SON VACIAS
        list_col0_hoja0 = list(df_PDF[0][0])
        if len(list_J_N) == 0:
            for i in list_col0_hoja0:
                if 'BRASIL' in list_leyenda_depurado_junto:
                    pais = 'BRA'
                    J_N = 'C.O'
                elif 'U.S.' in list_leyenda_depurado_junto:
                    pais = 'USA'
                    J_N = 'C.O'
                elif 'BRASIL' in str(i):
                    pais = 'BRA'
                    J_N = 'C.O'
                else:
                    pais = 'ZAF'
                    J_N = 'CUPO'

            list_J_N = [J_N]*len(list_auto_chasis)
            list_pais = [pais]*len(list_auto_chasis)

    # Completando listas, para que tengan el mismo num de filas
    list_factura = [factura]*len(list_J_N)
    list_Date = [date]*len(list_J_N)
    list_moneda_completa = list_moneda*len(list_J_N)
    list_leyenda_depurado_junto_completo = [list_leyenda_depurado_junto] + [np.nan]*(len(list_J_N) - 1)

    # Convirtiendo en DF
    df_factura = pd.DataFrame(list_factura, columns = ['Factura'])
    df_date = pd.DataFrame(list_Date, columns = ['Fecha'])
    df_auto = pd.DataFrame(list_auto, columns = ['Auto'])
    df_chasis = pd.DataFrame(list_chasis, columns = ['Chasis'])
    df_amount = pd.DataFrame(list_amount, columns = ['Amount'])
    df_J_N = pd.DataFrame(list_J_N, columns = ['J y N'])
    df_pais = pd.DataFrame(list_pais, columns = ['Pais'])
    df_moneda = pd.DataFrame(list_moneda_completa, columns = ['Moneda'])
    df_leyenda = pd.DataFrame(list_leyenda_depurado_junto_completo, columns = ['Leyenda'])

    # Concatenando df
    df_final = pd.concat([df_factura, df_pais, df_date, df_auto, df_chasis, df_J_N, df_amount, df_moneda, df_leyenda], axis = 1)

    return df_final



df_audi_vw = PDFs_v1('/Users/jorgevilchis/Documents/Cosas Gestell/PruebaFacturasAudi/20295611.PDF')

SyntaxError: invalid syntax (3462569068.py, line 1)

In [1]:
import random
import plotly.graph_objects as go

brands2 = [f"Marca {i}" for i in range(1, 8)]
counts2 = [random.randint(5, 50) for _ in brands2]
monthname = "Septiembre"

# Pick a color palette
colors = [
    "crimson", "royalblue", "darkorange", "seagreen",
    "purple", "gold", "teal"
]

fig = go.Figure()

for brand, count, color in zip(brands2, counts2, colors):
    fig.add_trace(go.Bar(
        x=[count],
        y=[brand],
        orientation='h',
        name=brand,
        marker=dict(color=color),
        text=[count],
        textposition="outside"
    ))

fig.update_layout(
    title=f"Estadística Mensual {monthname}",
    title_font=dict(size=20, color="#2c3e50"),
    plot_bgcolor="#f8f9fa",
    paper_bgcolor="#ffffff",
    xaxis=dict(showgrid=True, gridcolor="#e9ecef", title="Cantidad"),
    yaxis=dict(showgrid=False, title="Marca"),
    width=600,
    height=400,
    showlegend=False  # hide redundant legend
)

fig.show()


In [3]:
import plotly.express as px
import random

# Random demo data
brands2 = [f"Marca {i}" for i in range(1, 8)]
counts2 = [random.randint(5, 50) for _ in brands2]
monthname = "Septiembre"

# Create chart
fig = px.bar(
    x=counts2,
    y=brands2,
    orientation='h',
    title=f"Estadística Mensual {monthname}",
    labels={"x": "Cantidad", "y": "Marca"},
    text=counts2,
    color=brands2,
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textposition="outside")

# Export as HTML file
fig.write_html("static/plotly_chart.html", full_html=True, include_plotlyjs="cdn")


In [4]:
import plotly.express as px
import random

# Random demo data
brands2 = [f"Marca {i}" for i in range(1, 8)]
counts2 = [random.randint(5, 50) for _ in brands2]
monthname = "Septiembre"

# Radar chart
fig = px.line_polar(
    r=counts2,
    theta=brands2,
    line_close=True,
    title=f"Estadística Mensual {monthname}",
)

# Style tweaks
fig.update_traces(fill='toself', line_color="royalblue")
fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, max(counts2) + 5])
    ),
    showlegend=False,
    title_font=dict(size=20, color="#2c3e50"),
    paper_bgcolor="#ffffff"
)

fig.show()


In [5]:
import plotly.express as px
import random

brands2 = [f"Marca {i}" for i in range(1, 8)]
counts2 = [random.randint(5, 50) for _ in brands2]
monthname = "Septiembre"

# Radial bar chart
fig = px.bar_polar(
    r=counts2,
    theta=brands2,
    color=brands2,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f"Estadística Mensual {monthname}",
)

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, max(counts2) + 5])
    ),
    title_font=dict(size=20, color="#2c3e50"),
    paper_bgcolor="#ffffff"
)

fig.show()


In [6]:
fig = px.treemap(
    names=brands2,
    parents=[""] * len(brands2),  # flat structure
    values=counts2,
    title="Treemap de Marcas",
    color=counts2,
    color_continuous_scale="Blues"
)
fig.show()


In [7]:
fig = px.sunburst(
    names=brands2,
    parents=[""] * len(brands2),
    values=counts2,
    title="Sunburst de Marcas",
    color=counts2,
    color_continuous_scale="Viridis"
)
fig.show()


In [8]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=counts2,
    y=brands2,
    orientation="h",
    marker_color="lightblue",
    opacity=0.6
))

fig.add_trace(go.Scatter(
    x=counts2,
    y=brands2,
    mode="markers",
    marker=dict(color="blue", size=12),
    name="counts"
))

fig.update_layout(
    title="Lollipop Chart de Marcas",
    xaxis_title="Cantidad",
    yaxis_title="Marca"
)

fig.show()


In [44]:
###PAPEL DE TRABAJO INTERMEDIO 
import pandas as pd


#Se cargan los archivos (papel que nos mandan estos vergas y la extraccion historica)

suyo =  '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Diciembre 2025/NuevoFormato_Diciembre (SuyoSinLlenar).xls'
dfSuyo = pd.read_excel(suyo)
historico =  '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Diciembre 2025/RESULTADOs/ExtraccionHistorica_25Dic2025.xlsx'
dfHistorico = pd.read_excel(historico)

#Hacemos un join quedandonos solo con las columnas: VIN, FACTURA, FECHA FACTURA, PAIS, PREFERENCIA ARANCELARIA
print(dfSuyo.head(5))
print('------------------')
print(dfHistorico.head(5))


                 VIN   FACTURA  FECHA FACTURA  PAIS  PREFERENCIA ARANCELARIA
0  WAUREAGB1TR020101  20503090            NaN   NaN                      NaN
1  WAUAACGH4SA033303  20482852            NaN   NaN                      NaN
2  WAUCBJGF6SA086206  20482852            NaN   NaN                      NaN
3  WAUAYEGY1TA028021  20482850            NaN   NaN                      NaN
4  WAUAYEGY3TA018591  20468965            NaN   NaN                      NaN
------------------
   Factura Pais   Fecha    Auto             Chasis J y N      Amount Moneda  \
0  970601E  HUN   31125  KP1BC5  VSSAAAKP3T1022380     J  473.655,28    MXN   
1  968631E  HUN  311025  KP1CYS  VSSBBAKP5T1021989     J  639.191,17    MXN   
2  966904E  HUN  301025  KP1C1Y  VSSBCAKP4T1021589     J  595.417,00    MXN   
3  966843E  HUN  301025  KP1CYS  VSSBBAKP7T1021380     J  613.612,20    MXN   
4  967939E  HUN  301025  KP1CYS  VSSBBAKP2T1021674     J  602.649,78    MXN   

                                            

In [45]:
print(dfHistorico.columns)
print('---')
dfSuyo.columns

Index(['Factura', 'Pais', 'Fecha', 'Auto', 'Chasis', 'J y N', 'Amount',
       'Moneda', 'Leyenda', 'Filename'],
      dtype='object')
---


Index(['VIN', 'FACTURA', 'FECHA FACTURA', 'PAIS', 'PREFERENCIA ARANCELARIA'], dtype='object')

In [46]:
dfSuyo.drop(columns=['FECHA FACTURA','PAIS','PREFERENCIA ARANCELARIA'], inplace=True)
dfHistorico.rename (columns={'Factura':'FACTURA','Chasis':'VIN','Fecha':'FECHA FACTURA','Pais':'PAIS','J y N':'PREFERENCIA ARANCELARIA'}, inplace=True)


In [47]:
dfllenado = pd.merge(dfSuyo, dfHistorico[['VIN','FACTURA','FECHA FACTURA','PAIS','PREFERENCIA ARANCELARIA']], how='left', on='VIN')
dfllenado.to_excel('/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Diciembre 2025/PapelTrabajo_Intermedio_Diciembre31_NUEVO.xlsx', index=False)
dfllenado.head()

,VIN,FACTURA_x,FACTURA_y,FECHA FACTURA,PAIS,PREFERENCIA ARANCELARIA
0,WAUREAGB1TR020101,20503090,20503090,291025.0,ESP,J
1,WAUAACGH4SA033303,20482852,20482852,260925.0,DEU,N
2,WAUCBJGF6SA086206,20482852,20482852,260925.0,DEU,N
3,WAUAYEGY1TA028021,20482850,20482850,260925.0,DEU,J
4,WAUAYEGY3TA018591,20468965,20468965,180925.0,DEU,J


In [1]:
from pathlib import Path
import hashlib, os, shutil

def hash_file(filepath, chunk_size=8192):
    """Compute SHA256 hash for a given file."""
    sha = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(chunk_size):
            sha.update(chunk)
    return sha.hexdigest()

def find_unique_pdfs(root_dir, dry_run=True):
    root = Path(root_dir).resolve()
    unique_dir = root / "UniquePDFs"
    unique_dir.mkdir(exist_ok=True)

    seen_hashes = {}
    duplicates = []
    unique_files = []

    print(f"🔍 Scanning for PDF files under: {root}\n")

    for dirpath, _, filenames in os.walk(root):
        for filename in filenames:
            if not filename.lower().endswith(".pdf"):
                continue

            filepath = Path(dirpath) / filename
            try:
                file_hash = hash_file(filepath)
            except Exception as e:
                print(f"⚠️ Could not hash {filepath}: {e}")
                continue

            if file_hash in seen_hashes:
                duplicates.append(filepath)
            else:
                seen_hashes[file_hash] = filepath
                unique_files.append(filepath)

    print(f"\n✅ Found {len(unique_files)} unique PDFs")
    print(f"⚠️ Found {len(duplicates)} duplicates\n")

    if dry_run:
        print("💡 Dry run mode: No files will be moved.\n")
    else:
        print(f"📂 Moving unique files to: {unique_dir}\n")
        for f in unique_files:
            try:
                dest = unique_dir / f.name
                counter = 1
                while dest.exists():
                    dest = unique_dir / f"{dest.stem}_{counter}{dest.suffix}"
                    counter += 1
                shutil.move(str(f), dest)
            except Exception as e:
                print(f"⚠️ Could not move {f}: {e}")
        print("\n✅ Move completed.")

    return unique_files, duplicates


In [2]:
path = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Octubre2025/FacturasPeriodoOctubre'

unique, duplicates = find_unique_pdfs(path, dry_run=False)


🔍 Scanning for PDF files under: /home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Octubre2025/FacturasPeriodoOctubre


✅ Found 1031 unique PDFs
⚠️ Found 1025 duplicates

📂 Moving unique files to: /home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Octubre2025/FacturasPeriodoOctubre/UniquePDFs


✅ Move completed.


In [66]:
##GRAFICAS Anuales

import pandas as pd


#Se cargan los archivos (papel que nos mandan estos vergas y la extraccion historica)


Concentrado2 =  '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/Diciembre 2025/RESULTADOs/Historico Latest7Enero/Concentrado2_HistoricoNuevo.xlsx'
dfHistorico = pd.read_excel(Concentrado2)
print(dfHistorico.head(10))

   ID    AUTO      FACT  FECFACT             CHASIS      PRECIO  \
0  FA  SBHA14  96984228    61124  WV1DDASK0RX094028  343.207,00   
1  FA  SBHA14  96984228    61124  WV1DDASK3RX094041  343.207,00   
2  FA  SBHA14  96984228    61124  WV1DDASK8RX093905  343.207,00   
3  FA  SBHA14  96984228    61124  WV1DDASK9RX093881  343.207,00   
4  FA  SBHA14  96983063    51124  WV1DDASK4RX093707  343.207,00   
5  FA  SBHA14  96983063    51124  WV1DDASK5RX093702  343.207,00   
6  FA  SBHA14  96984228    61124  WV1DDASK2RX093706  343.207,00   
7  FA  SBHA14  96986116   121124  WV1DDASK4RX094016  343.207,00   
8  FA  SBHA14  96986116   121124  WV1DDASK4RX093979  343.207,00   
9  FA  SBHA14  96986116   121124  WV1DDASK0RX093980  343.207,00   

                       TIPO      FRACCION PAIS  PATENTE  PEDIMENTO  \
0  CADDY 5 Cargo Van Diesel  8.703330e+09  POL     6120    4000455   
1  CADDY 5 Cargo Van Diesel  8.703330e+09  POL     6120    4000455   
2  CADDY 5 Cargo Van Diesel  8.703330e+09  POL     6

In [67]:
import pandas as pd
import plotly.express as px

# Copia defensiva
df = dfHistorico.copy()

# --- Fechas ---
# FECFACT parece venir como entero tipo DDMMYY o similar
df['FECFACT'] = pd.to_datetime(df['FECFACT'].astype(str), format='%d%m%y', errors='coerce')

# FECHA PEDIMENTO ya está en formato YYYYMMDD
df['FECHA PEDIMENTO'] = pd.to_datetime(df['FECHA PEDIMENTO'].astype(str), format='%Y%m%d', errors='coerce')

# --- Columna J y N ---
df['J y N'] = df['J y N'].str.strip()

# --- Asegurar strings ---
df['MARCA'] = df['MARCA'].astype(str)
df['PAIS'] = df['PAIS'].astype(str)
df['TIPO'] = df['TIPO'].astype(str)


In [68]:
jn_color_map = {
    'J': '#16a34a',      # Verde
    'N': '#dc2626',      # Rojo
    'Cupo': '#facc15'    # Amarillo
}

In [69]:
pie_data = df['J y N'].value_counts().reset_index()
pie_data.columns = ['J y N', 'Cantidad']

fig1 = px.pie(
    pie_data,
    names='J y N',
    values='Cantidad',
    title='Proporción de Calificación J vs N [Anual]' ,
    color='J y N',
    color_discrete_map=jn_color_map
)

fig1.show()


In [70]:
ts_fecfact_total = (
    df.groupby('FECFACT')
      .size()
      .reset_index(name='Cantidad')
)

fig2 = px.line(
    ts_fecfact_total,
    x='FECFACT',
    y='Cantidad',
    title='Total de Operaciones por Fecha de Factura [Anual]'
)

fig2.show()


In [71]:
ts_fecfact_marca = (
    df.groupby(['FECFACT', 'MARCA'])
      .size()
      .reset_index(name='Cantidad')
)

fig3 = px.line(
    ts_fecfact_marca,
    x='FECFACT',
    y='Cantidad',
    color='MARCA',
    title='Operaciones por Fecha de Factura y Marca [Anual]'
)

fig3.show()


In [72]:
bar_pais_jn = (
    df.groupby(['PAIS', 'J y N'])
      .size()
      .reset_index(name='Cantidad')
)



fig6 = px.bar(
    bar_pais_jn,
    x='PAIS',
    y='Cantidad',
    color='J y N',
    barmode='group',
    title='Cantidad de Operaciones por País y Calificación [Anual]',
    color_discrete_map=jn_color_map
)

fig6.show()



In [73]:
treemap_data = (
    df.groupby(['MARCA', 'TIPO'])
      .size()
      .reset_index(name='Cantidad')
)

fig7 = px.treemap(
    treemap_data,
    path=['MARCA', 'TIPO'],
    values='Cantidad',
    title='Distribución por Marca y Tipo de Vehículo [Anual]'
)

fig7.show()


In [74]:
ts_pedimento_marca = (
    df.groupby(['FECHA PEDIMENTO', 'MARCA'])
      .size()
      .reset_index(name='Cantidad')
)

fig5 = px.line(
    ts_pedimento_marca,
    x='FECHA PEDIMENTO',
    y='Cantidad',
    color='MARCA',
    title='Operaciones por Fecha de Pedimento y Marca [Anual]'
)

fig5.show()


In [75]:
ts_pedimento_total = (
    df.groupby('FECHA PEDIMENTO')
      .size()
      .reset_index(name='Cantidad')
)

fig4 = px.line(
    ts_pedimento_total,
    x='FECHA PEDIMENTO',
    y='Cantidad',
    title='Total de Operaciones por Fecha de Pedimento [Anual]'
)

fig4.show()


In [5]:
import pandas as pd

#Archivos a utilizar para la revision con el listado del barco:
#-Concentrado2
concentrado2 = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Concentrado2_SoloEnero2026.xlsx'

#-FacturasFaltantes
facturasFaltantes = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/Facturas_Faltantes_SoloEnero2026.xlsx'

#-ExtraccionHistorica
Extraccion = '/Users/jorgevilchis/Documents/Cosas Gestell/Aduanas1/P1 - PDFs/2026/Enero/ExtraccionGeneral2Enero2026.xlsx'
#-ChasisBarcoEJEMPLO
chasisBarco = '/Users/jorgevilchis/Downloads/chasis de barco SIEM CICERO 30.09.2025.xlsx'


dfConcentrado2 = pd.read_excel(concentrado2)
dfFacturasFaltantes = pd.read_excel(facturasFaltantes)
dfExtraccion = pd.read_excel(Extraccion)
dfChasisBarco = pd.read_excel(chasisBarco)
#Queremos ver que tenemos, que no, y las que no en donde es que estaban. 

print('Concentrado2:')
dfConcentrado2.head(10)
print('---'*20)
print('FacturasFaltantes:')
dfFacturasFaltantes.head(3)
print('---'*20)
print('ExtraccionHistorica:')
dfExtraccion.head(3)
print('---'*20) 
print('ChasisBarco:')
dfChasisBarco.head(3)
print('---'*20)


Concentrado2:
------------------------------------------------------------
FacturasFaltantes:
------------------------------------------------------------
ExtraccionHistorica:
------------------------------------------------------------
ChasisBarco:
------------------------------------------------------------


In [8]:
dfChasisBarco.head()

,Chassis Number,Model,Ship Name,FI: Invoice No.,Estatus
0,VSSAAAKP2T1007823,KP1BC5,SIEM CICERO,879120E,ok
1,VSSAAAKP7T1007834,KP1BC5,SIEM CICERO,879121E,ok
2,VSSBCAKP7T1007671,KP1C1Y,SIEM CICERO,879122E,ok
3,VSSAAAKP3T1007832,KP1BC5,SIEM CICERO,879124E,ok
4,VSSAAAKP4T1007810,KP1BC5,SIEM CICERO,879125E,ok


In [9]:
dfConcentrado2.head()

,ID,AUTO,FACT,FECFACT,CHASIS,PRECIO,TIPO,FRACCION,PAIS,PATENTE,PEDIMENTO,FECHA PEDIMENTO,FLETES,SEGUROS,ADUANA,MARCA,J y N,PAIS_WITNESS
0,FA,8YMRWY,20468965,180925,WUAB3FGY5TA902045,"943.366,00",Audi RS 3 Sedan,8703239900,DEU,6120,6000026,20260209,0.0,0.274388,NaN,AUDI,J,sin cambio
1,FA,8YMRWY,20506331,41125,WUAB3FGY5TA904538,"977.006,00",Audi RS 3 Sedan,8703239900,DEU,6120,6000026,20260209,0.0,0.274388,NaN,AUDI,J,sin cambio
2,FA,8YMRWY,20486468,21025,WUAB3FGY7TA903018,"1.088.214,00",Audi RS 3 Sedan,8703239900,DEU,6120,6000026,20260209,0.0,0.274388,NaN,AUDI,J,sin cambio
3,FA,8YMRWY,20537854,241225,WUAB3FGY7TA907991,"964.268,00",Audi RS 3 Sedan,8703239900,DEU,6120,6000026,20260209,0.0,0.274388,NaN,AUDI,J,sin cambio
4,FA,8YMRWY,20488876,71025,WUAB3FGY2TA903721,"1.010.482,00",Audi RS 3 Sedan,8703239900,DEU,6120,6000026,20260209,0.0,0.274388,NaN,AUDI,J,sin cambio
